# 04 Coregistration

This notebook is part of the separate anatomy/geometry preparation block. It launches the MNE coregistration GUI for selected recordings and helps keep track of expected `*-trans.fif` files.

Coregistration links the MEG head-coordinate system from one raw recording to the subject's FreeSurfer MRI anatomy. It can be done after FreeSurfer reconstruction and before forward/inverse modeling.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import (
    iter_recordings, 
    selected_recordings_to_dataframe,
    should_overwrite,
    recording_label
)
from meeg_pipeline.anatomy import (
    coregistration_status_to_dataframe,
    first_recording_per_subject,
    launch_coregistration_gui,
    results_to_dataframe,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Settings

By default, this notebook selects one representative recording per subject. Coregistration itself is an interactive, manual step. The launch cell below opens the MNE coregistration GUI for one selected table row, not as a batch loop.

The transform filename is controlled by the project config entry `anatomy.coregistration.transform_scope`:

- `subject`: one transform per subject, shared across tasks/sessions where appropriate
- `session`: one transform per subject/session
- `recording`: one transform per full subject/session/task/run recording

For many single-session projects, `subject` is the right setting when all tasks for a participant share the same head-MRI transform.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

# Which recordings should open the GUI? This is independent of the saved
# transform scope configured in configs/local.yaml.
COREGISTRATION_MODE = "first_recording_per_subject"  # "first_recording_per_subject" | "all_recordings"

TRANS_DESC = "coreg"
TRANS_SCOPE = config.anatomy.coregistration.transform_scope
ALLOW_COMPATIBLE_TRANS_FALLBACK = config.anatomy.coregistration.allow_compatible_fallback

# Existing transforms are skipped by default. Use ["coregistration"] to force
# overwriting/re-saving them.
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "coregistration_mode": COREGISTRATION_MODE,
            "transform_scope": TRANS_SCOPE,
            "allow_compatible_fallback": ALLOW_COMPATIBLE_TRANS_FALLBACK,
            "trans_desc": TRANS_DESC,
            "overwrite_coregistration": should_overwrite(
                "coregistration",
                OVERWRITE_STEPS,
            ),
        }
    ]
)


## Select recordings

The selected recordings provide the raw BIDS `inst` files for the MNE coregistration GUI.


In [ ]:
all_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

if COREGISTRATION_MODE == "first_recording_per_subject":
    selected_recordings = first_recording_per_subject(all_recordings)
elif COREGISTRATION_MODE == "all_recordings":
    selected_recordings = all_recordings
else:
    raise ValueError(
        "COREGISTRATION_MODE must be 'first_recording_per_subject' or 'all_recordings'."
    )

selected_recordings_to_dataframe(selected_recordings)

## Coregistration input/output status

Check that the raw BIDS file and the FreeSurfer subject exist before opening the GUI. The `trans_path` column is the canonical path where the transform should be saved from the GUI. The entity scope of that path comes from `anatomy.coregistration.transform_scope`.


In [ ]:
coreg_status = coregistration_status_to_dataframe(
    config,
    selected_recordings,
    trans_desc=TRANS_DESC,
)

coreg_status

## Open coregistration GUI(s)

This cell opens the MNE coregistration GUI for the selected recordings.

The raw BIDS file is passed as `inst`, so the matching digitization, head-shape, and HPI information is loaded automatically for each recording.

The GUI is opened with `block=True`. This means that only one GUI window is active at a time. After you close the current GUI window, the notebook automatically saves the current transform to the printed `Transform target` path and then continues with the next pending recording.

You do not need to use the GUI save button. Perform the coregistration, close the GUI window, ignore the GUI save warning if it appears, and let the notebook write the transform automatically.

## Saving the coregistration transform

In this notebook, the transform is saved automatically after the coregistration GUI is closed.

Workflow:

1. Open the GUI.
2. Perform the coregistration.
3. Do not use the GUI save button.
4. Close the GUI window.
5. If the GUI asks whether you want to save before closing, you can ignore/dismiss this warning.
6. The notebook writes the current transform automatically to the printed `Transform target` path.

This avoids manually choosing the output path for each subject or recording. The notebook checks after closing whether the expected `*_trans.fif` file exists.

## Coregistration quality guidelines

Use the numbers in the GUI as practical quality-control indicators. They should not be interpreted as absolute pass/fail rules, but they are useful for deciding whether a transform is good enough or needs manual adjustment.

### Fiducial distances

The fiducial distances describe the mismatch between the anatomical fiducials and the digitized fiducial points, usually reported for LPA, nasion, and RPA.

Example:

`8.5, 1.6, 7.0 mm`

Guidelines:

- Ideal: below 5 mm.
- Acceptable: up to 10 mm if the head-shape fit is good.
- If one fiducial is clearly higher than the others, adjust the fiducial point in the MRI or in the digitized head shape if possible.

Interpretation of the example: one fiducial at 8.5 mm is somewhat high, but still usually acceptable if the HSP fit is good. It may reflect imprecise fiducial placement or a small mismatch between MRI anatomy and digitized head shape.

### HSP + HPI fit after ICP

This value summarizes the average distance of the head-shape points and HPI points after the ICP fit.

Example:

`104 HSP + HPI: 1.4 ± 1.0 mm`

Guidelines:

- Mean HSP/HPI distance below 2 mm is very good.
- A value around 1.4 ± 1.0 mm indicates an excellent fit.

### HSP-to-MRI surface distance

This compares the digitized head-shape points to the reconstructed MRI-derived scalp surface.

Example:

`mean / min / max: 1.50 / 0.17 / 8.50 mm`

Guidelines:

- Mean below 2 mm is very good.
- Maximum values below about 10 mm are usually acceptable when they represent only a few outlier points.
- If the mean is good but the max is high, inspect the outlier points and delete clearly wrong head-shape points in the GUI.

Interpretation of the example: a mean of 1.5 mm is excellent. A max of 8.5 mm likely reflects one or a few outlier points and is usually acceptable after visual inspection.

### Practical decision rule

A coregistration is usually acceptable when:

- the head shape visually follows the MRI-derived scalp surface,
- the mean HSP-to-MRI distance is below about 2 mm,
- fiducials are mostly below 5 mm or at least below 10 mm,
- larger maximum errors are limited to a few obvious outlier points,
- the sensor helmet position looks anatomically plausible.


In [ ]:
from pathlib import Path

import mne
import pandas as pd

mne.viz.set_3d_backend("pyvistaqt")

RUN_COREGISTRATION_GUI = True
MAX_COREGISTRATION_GUIS = None
HEAD_HIGH_RES = True

SAVE_TRANS_AFTER_CLOSE = "confirm"  # "confirm" | "always" | "never"
CONFIRM_SAVE_WORD = "y"

overwrite_coregistration = should_overwrite(
    "coregistration",
    OVERWRITE_STEPS,
)


def freesurfer_subject_label(subject):
    subject = str(subject)
    return subject if subject.startswith("sub-") else f"sub-{subject}"


status_table = coregistration_status_to_dataframe(
    config,
    selected_recordings,
    trans_desc=TRANS_DESC,
)

display(status_table)

pending = []

for idx, recording in enumerate(selected_recordings):
    row = status_table.iloc[idx]

    if bool(row["trans_exists"]) and not overwrite_coregistration:
        print(f"Skipping existing transform: {row['trans_path']}")
        continue

    pending.append((idx, recording, row))

if MAX_COREGISTRATION_GUIS is not None:
    pending = pending[:MAX_COREGISTRATION_GUIS]

results = []

if not RUN_COREGISTRATION_GUI:
    print("Coregistration GUI disabled.")
elif not pending:
    print("No pending coregistration transforms.")
else:
    subjects_dir = Path(config.freesurfer.subjects_dir).expanduser().resolve()

    for idx, recording, row in pending:
        bids_subject = str(recording["subject"])
        fs_subject = freesurfer_subject_label(bids_subject)

        inst_path = Path(row["raw_path"]).expanduser().resolve()
        trans_path = Path(row["trans_path"]).expanduser().resolve()
        trans_path.parent.mkdir(parents=True, exist_ok=True)

        head_files = [
            subjects_dir / fs_subject / "bem" / f"{fs_subject}-head.fif",
            subjects_dir / fs_subject / "bem" / f"{fs_subject}-head-medium.fif",
            subjects_dir / fs_subject / "bem" / f"{fs_subject}-head-sparse.fif",
            subjects_dir / fs_subject / "bem" / f"{fs_subject}-head-dense.fif",
        ]
        existing_head_files = [path for path in head_files if path.exists()]

        if not existing_head_files:
            raise FileNotFoundError(
                "No head surface found for FreeSurfer subject "
                f"{fs_subject} in {subjects_dir / fs_subject / 'bem'}"
            )

        print("\n" + "=" * 80)
        print(f"Coregistration row:  {idx}")
        print(f"BIDS subject:        {bids_subject}")
        print(f"FreeSurfer subject:  {fs_subject}")
        print(f"SUBJECTS_DIR:        {subjects_dir}")
        print(f"Inst/raw file:       {inst_path}")
        print(f"Transform target:    {trans_path}")
        print("Existing head files:")
        for path in existing_head_files:
            print(f"  {path.name}")
        print("=" * 80)

        coreg_ui = mne.gui.coregistration(
            subject=fs_subject,
            subjects_dir=str(subjects_dir),
            inst=str(inst_path),
            trans=str(trans_path) if trans_path.exists() else None,
            head_high_res=HEAD_HIGH_RES,
            show=True,
            block=True,
        )

        saved = False
        error = None

        if SAVE_TRANS_AFTER_CLOSE == "always":
            should_save = True

        elif SAVE_TRANS_AFTER_CLOSE == "never":
            should_save = False

        elif SAVE_TRANS_AFTER_CLOSE == "confirm":
            print()
            print("=" * 80)
            print("Coregistration GUI was closed.")
            print(f"Recording: {recording_label(recording)}")
            print(f"Target transform path:")
            print(f"  {trans_path}")
            print()
            print("Save the current GUI transform to this file?")
            response = input(f"Save transform? Type {CONFIRM_SAVE_WORD!r} to save, anything else to skip.").strip()
            should_save = response == CONFIRM_SAVE_WORD

        else:
            raise ValueError(
                "SAVE_TRANS_AFTER_CLOSE must be one of: "
                "'confirm', 'always', 'never'."
            )

        if should_save:
            try:
                trans_path.parent.mkdir(parents=True, exist_ok=True)

                mne.write_trans(
                    trans_path,
                    coreg_ui.coreg.trans,
                    overwrite=True,
                )

                saved = True
                print(f"Transform saved to: {trans_path}")

            except Exception as exc:
                error = repr(exc)
                print(f"Could not save transform: {error}")

        else:
            print("Transform was not saved.")

        exists_after = trans_path.exists()

        exists_after = trans_path.exists()

        results.append(
            {
                "index": idx,
                "bids_subject": bids_subject,
                "freesurfer_subject": fs_subject,
                "inst_path": str(inst_path),
                "trans_path": str(trans_path),
                "auto_saved": saved,
                "trans_exists_after": exists_after,
                "error": error,
            }
        )

        display(pd.DataFrame(results))

pd.DataFrame(results)

## Final status

Run this after saving transforms to confirm that the expected files exist.


In [ ]:
coregistration_status_to_dataframe(
    config,
    selected_recordings,
    trans_desc=TRANS_DESC,
)